# VEHMS-DL: Analysis Report Implementation

## Deep Learning Enhanced Vehicular Engine Health Monitoring System

This notebook implements the comprehensive analysis report based on the IMPLEMENTATION_PLAN.md and README.md.

### Target Performance Metrics
- **Accuracy**: 97.5-98.5% (current best: 96.85%)
- **AUC**: 0.999+ (current best: 0.9984)
- **RMSE**: 0.28-0.32 (current best: 0.3413)

---
## Phase 1: Setup and Baseline Reproduction

In [ ]:
# Core imports and VEHMS modules
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from vehms import (
    DataLoader, DataCleaner, DataPreprocessor, BaseClassifierModule,
    StackedEnsemble, ExistingResearchStackedEnsemble, DynamicClassifierSelector,
    ModelEvaluator, PerformanceVisualizer, XAIExplainer, RANDOM_SEED
)

np.random.seed(RANDOM_SEED)
print(f"VEHMS-DL Analysis Report | Random Seed: {RANDOM_SEED}")

In [ ]:
# Load and preprocess augmented dataset
loader = DataLoader()
df = loader.load_dataset('dataset/augmented_data_with_environment.csv')
loader.validate_columns(df)
loader.validate_data_types(df)

# Preprocess data
preprocessor = DataPreprocessor()
X, y = preprocessor.separate_features_target(df)
X_scaled = preprocessor.fit_transform_features(X)
y_encoded = preprocessor.encode_target(y)
X_train, X_test, y_train, y_test = preprocessor.train_test_split(X_scaled, y_encoded)

feature_names = preprocessor.feature_names
class_names = list(preprocessor.label_encoder.classes_)

In [ ]:
# Train base classifiers
base_module = BaseClassifierModule()
base_results = base_module.train_all(X_train, y_train, X_test, y_test)
display(base_results)

---
## Phase 2: Stacked Ensemble Models

In [ ]:
# Proposed Stacked Models
stacked = StackedEnsemble(cv=5, random_state=RANDOM_SEED)

stacked.train_stacked_model('Stacked Model 1', stacked.create_stacked_model_1(), X_train, y_train)
stacked.train_stacked_model('Stacked Model 2', stacked.create_stacked_model_2(), X_train, y_train)
stacked.train_stacked_model('Stacked Model 3', stacked.create_stacked_model_3(), X_train, y_train)

stacked.display_architecture('Stacked Model 1')

In [ ]:
# Existing Research Stacked Models
er_stacked = ExistingResearchStackedEnsemble(cv=5, random_state=RANDOM_SEED)

er_stacked.train_stacked_model('ER-Stacked Model 1', er_stacked.create_er_stacked_model_1(), X_train, y_train)
er_stacked.train_stacked_model('ER-Stacked Model 2', er_stacked.create_er_stacked_model_2(), X_train, y_train)
er_stacked.train_stacked_model('ER-Stacked Model 3', er_stacked.create_er_stacked_model_3(), X_train, y_train)

er_stacked.display_architecture('ER-Stacked Model 3')

---
## Phase 3: Dynamic Classifier Selection

In [ ]:
# Dynamic Classifier Selection with Q-statistic diversity
selector = DynamicClassifierSelector(cv=5, random_state=RANDOM_SEED)
selection_results = selector.run_all_selection_methods(X_train, y_train, X_test, y_test, top_k=5)

In [ ]:
# Create and train dynamic stacks
ds_stacks = {}
for method, selected in selection_results.items():
    stack = selector.create_dynamic_stack(selected)
    stack.fit(X_train, y_train)
    ds_stacks[f'DS-Stack {method.title()}'] = {'model': stack, 'classifiers': selected}
    print(f"DS-Stack {method.title()}: {selected}")

---
## Phase 4: Deep Learning Integration

In [ ]:
from vehms.deep_learning_classifiers import (
    CNNClassifier, LSTMClassifier, GRUClassifier, 
    CNNLSTMClassifier, AttentionLSTMClassifier
)

n_features = X_train.shape[1]
n_classes = len(np.unique(y_train))

# Deep Learning Classifiers
dl_classifiers = {
    'CNN': CNNClassifier(n_features=n_features, n_classes=n_classes, epochs=50, verbose=0),
    'LSTM': LSTMClassifier(n_features=n_features, n_classes=n_classes, epochs=50, verbose=0),
    'GRU': GRUClassifier(n_features=n_features, n_classes=n_classes, epochs=50, verbose=0),
    'CNN-LSTM': CNNLSTMClassifier(n_features=n_features, n_classes=n_classes, epochs=50, verbose=0),
    'Attention-LSTM': AttentionLSTMClassifier(n_features=n_features, n_classes=n_classes, epochs=50, verbose=0)
}

dl_results = {}
for name, clf in dl_classifiers.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train, y_train)
    dl_results[name] = clf
    print(f"{name} trained. Test accuracy: {clf.score(X_test, y_test):.4f}")

---
## Phase 5: Comprehensive Model Evaluation

In [ ]:
evaluator = ModelEvaluator()

# Evaluate base classifiers
for name, clf in base_module.get_all_classifiers().items():
    evaluator.evaluate_model(clf, X_test, y_test, name)

# Evaluate stacked models
for name, model in stacked.get_all_models().items():
    evaluator.evaluate_model(model, X_test, y_test, name)

# Evaluate ER stacked models
for name, model in er_stacked.get_all_models().items():
    evaluator.evaluate_model(model, X_test, y_test, name)

# Evaluate dynamic stacks
for name, data in ds_stacks.items():
    evaluator.evaluate_model(data['model'], X_test, y_test, name)

# Evaluate DL classifiers
for name, clf in dl_results.items():
    evaluator.evaluate_model(clf, X_test, y_test, f'DL-{name}')

In [ ]:
# Generate comparison table
comparison_df = evaluator.compare_models()
print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*80)
display(comparison_df)

---
## Phase 6: Performance Visualization

In [ ]:
visualizer = PerformanceVisualizer()

# Performance comparison charts
visualizer.plot_metrics_comparison(comparison_df)
visualizer.plot_model_ranking(comparison_df)

In [ ]:
# Best model confusion matrix
best_model_name = comparison_df.iloc[0]['Model']
best_metrics = evaluator.metrics_results[best_model_name]
visualizer.plot_confusion_matrix(best_metrics['Confusion_Matrix'], class_names, best_model_name)

---
## Phase 7: Explainable AI Analysis

In [ ]:
# Get best performing model for XAI
best_model = None
if best_model_name in stacked.get_all_models():
    best_model = stacked.get_model(best_model_name)
elif best_model_name in er_stacked.get_all_models():
    best_model = er_stacked.get_model(best_model_name)
elif best_model_name.startswith('DS-Stack'):
    best_model = ds_stacks[best_model_name]['model']
else:
    best_model = base_module.get_classifier(best_model_name)

print(f"Best Model: {best_model_name}")

In [ ]:
# Initialize XAI Explainer
explainer = XAIExplainer(best_model, feature_names, class_names)

# SHAP Analysis
explainer.initialize_shap(X_train, max_samples=100)
explainer.compute_shap_values(X_test[:50], nsamples=50)
explainer.plot_shap_summary()
explainer.plot_shap_bar(class_idx=0)

In [ ]:
# LIME Analysis
explainer.initialize_lime(X_train)
lime_exp = explainer.explain_instance_lime(X_test[0])
explainer.plot_lime_explanation(lime_exp)
explainer.display_lime_text(lime_exp)

---
## Phase 8: Target Metrics Verification

In [ ]:
# Verify against target metrics
print("\n" + "="*80)
print("TARGET METRICS VERIFICATION")
print("="*80)

verification = evaluator.verify_targets(best_model_name)
for metric, result in verification.items():
    status = "✓ PASSED" if result['passed'] else "✗ FAILED"
    print(f"{metric}: Target={result['target']:.4f}, Actual={result['actual']:.4f}, Deviation={result['deviation']:.2f}% [{status}]")

---
## Final Summary

In [ ]:
print("\n" + "="*80)
print("VEHMS-DL ANALYSIS REPORT SUMMARY")
print("="*80)

best = comparison_df.iloc[0]
print(f"\n🏆 Best Model: {best['Model']}")
print(f"   Accuracy:  {best['Accuracy']:.4f} ({best['Accuracy']*100:.2f}%)")
print(f"   Precision: {best['Precision']:.4f}")
print(f"   AUC:       {best['AUC']:.4f}")
print(f"   RMSE:      {best['RMSE']:.4f}")

print(f"\n📊 Total Models Evaluated: {len(comparison_df)}")
print(f"   - Base Classifiers: {len(base_module.get_all_classifiers())}")
print(f"   - Stacked Models: {len(stacked.get_all_models())}")
print(f"   - ER-Stacked Models: {len(er_stacked.get_all_models())}")
print(f"   - Dynamic Stacks: {len(ds_stacks)}")
print(f"   - Deep Learning: {len(dl_results)}")

print("\n" + "="*80)